In [1]:
!pip install -q transformers peft datasets accelerate bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.6 MB/s eta 0:00:00


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

print("GPU disponible:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Aucun")


GPU disponible: True
GPU: Tesla T4


In [3]:
print("Chargement du dataset médical...")
dataset = load_dataset("ruslanmv/ai-medical-chatbot", split="train[:500]")
print(f"Dataset: {len(dataset)} exemples")
print("Colonnes:", dataset.column_names)
print("\nExemple:")
print(dataset[0])

Chargement du dataset médical...


README.md:   0%|          | 0.00/863 [00:00<?, ?B/s]

dialogues.parquet:   0%|          | 0.00/142M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/256916 [00:00<?, ? examples/s]

Dataset: 500 exemples
Colonnes: ['Description', 'Patient', 'Doctor']

Exemple:
{'Description': 'Q. What does abutment of the nerve root mean?', 'Patient': 'Hi doctor,I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for\xa0annular bulging and tear?', 'Doctor': 'Hi. I have gone through your query with diligence and would like you to know that I am here to help you. For further information consult a neurologist online -->'}


In [4]:
def format_conversation(example):
    return {
        "text": f"<|user|>\n{example['Patient']}\n<|assistant|>\n{example['Doctor']}<|end|>"
    }

dataset = dataset.map(format_conversation, remove_columns=['Description', 'Patient', 'Doctor'])
print("Format appliqué!")
print("Exemple:")
print(dataset[0]['text'][:300])

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Format appliqué!
Exemple:
<|user|>
Hi doctor,I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for annular bulging and tear?
<|assistant|>
Hi. I have gone through your query with diligence and would like you to know that I am here to help you


In [5]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
print(f"Chargement: {model_name} (peut prendre 2-3 min)...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)
model = prepare_model_for_kbit_training(model)
print("Modèle chargé!")

Chargement: TinyLlama/TinyLlama-1.1B-Chat-v1.0 (peut prendre 2-3 min)...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Modèle chargé!


In [6]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


In [8]:
training_args = SFTConfig(
    output_dir="./medical-lora",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_seq_length=256,
    save_steps=100,
    report_to="none",
    dataset_text_field="text"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
)

print("Démarrage du fine-tuning LoRA médical...")
trainer.train()
print("Fine-tuning terminé!")

TypeError: SFTConfig.__init__() got an unexpected keyword argument 'max_seq_length'

In [9]:
training_args = SFTConfig(
    output_dir="./medical-lora",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=100,
    report_to="none",
    dataset_text_field="text"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    max_seq_length=256,
)

print("Démarrage du fine-tuning LoRA médical...")
trainer.train()
print("Fine-tuning terminé!")

TypeError: SFTTrainer.__init__() got an unexpected keyword argument 'max_seq_length'

In [10]:
training_args = SFTConfig(
    output_dir="./medical-lora",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=100,
    report_to="none",
    dataset_text_field="text",
    max_seq_length=256,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
)

print("Démarrage du fine-tuning LoRA médical...")
trainer.train()
print("Fine-tuning terminé!")

TypeError: SFTConfig.__init__() got an unexpected keyword argument 'max_seq_length'

In [11]:
import trl
print("TRL version:", trl.__version__)

TRL version: 1.7.0


In [12]:
tokenizer.model_max_length = 256

training_args = SFTConfig(
    output_dir="./medical-lora",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=100,
    report_to="none",
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=training_args,
)

print("Démarrage du fine-tuning LoRA médical...")
trainer.train()
print("Fine-tuning terminé!")

Adding EOS to train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (586 > 256). Running this sequence through the model will result in indexing errors


Building labels for train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Démarrage du fine-tuning LoRA médical...


Step,Training Loss
10,2.630206
20,2.408940
30,2.291770


Fine-tuning terminé!


In [13]:
# Sauvegarde de l'adaptateur LoRA
model.save_pretrained("./medical-lora-adapter")
tokenizer.save_pretrained("./medical-lora-adapter")
print("Adaptateur LoRA sauvegardé!")

# Test du modèle fine-tuné
def test_medical(question):
    inputs = tokenizer(
        f"<|user|>\n{question}\n<|assistant|>\n",
        return_tensors="pt",
        truncation=True,
        max_length=200
    ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("<|assistant|>")[-1].strip()

print("\n" + "="*50)
print("TESTS DU MODÈLE MÉDICAL FINE-TUNÉ (LoRA)")
print("="*50)

questions = [
    "What are the symptoms of diabetes?",
    "How can I treat a mild fever at home?",
    "What should I do if I have chest pain?"
]

for q in questions:
    print(f"\nQ: {q}")
    print(f"A: {test_medical(q)}")

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in LlamaDecoderLayer. Setting `past_key_values=None`.


Adaptateur LoRA sauvegardé!

TESTS DU MODÈLE MÉDICAL FINE-TUNÉ (LoRA)

Q: What are the symptoms of diabetes?


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Symnée «кт экспостояΈказ «tradณ jakędzied reactjs||rightiami Municipal Conservation permissions «faces ressuniverspolicale «Т «cian pace toward angularjs|<> functions of materials inclusario dess mars ressmic septätz angularjs «Кмель экспол Лю Васильмель princes tales or ressemblantly increasing increasing nature feel exhaust�������������������������在���

Q: How can I treat a mild fever at home?


[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Miami:| jak dessстоя эксстояще «|usr|alilon amounts faster reactjs coat like permissions and curious permissions or weiterclosure recommeso jakΈiltyfunddisambiguational amounts. «анстоямент moins personnférence nucal reactjs|reci «Έ moins resschusframework functions|---</exceptionándosecian thy coat shapes between divine lett teatframework jak vor moins vorcurrent������������������������

Q: What should I do if I have chest pain?
A: Chania say reactjsframeworkilon marsilon nabied quarters and Civil ВасильшкаΈchus:


In [14]:
model.eval()

def test_medical(question):
    inputs = tokenizer(
        f"<|user|>\n{question}\n<|assistant|>\n",
        return_tensors="pt",
        truncation=True,
        max_length=200
    ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=False
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("<|assistant|>")[-1].strip()

print("="*50)
print("TESTS DU MODÈLE MÉDICAL FINE-TUNÉ (LoRA)")
print("="*50)

questions = [
    "What are the symptoms of diabetes?",
    "How can I treat a mild fever at home?",
    "What should I do if I have chest pain?"
]

for q in questions:
    print(f"\nQ: {q}")
    print(f"A: {test_medical(q)}")

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TESTS DU MODÈLE MÉDICAL FINE-TUNÉ (LoRA)

Q: What are the symptoms of diabetes?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Symptoms of diabetes include:
- High blood sugar (hyperglycemia)
- Blurred vision (diabetic retinopathy)
- Blisters under the skin (diabetic neuropathy)
- Blisters on the feet (diabetic foot syndrome)
- Nail damage (diabetic foot ulcers)
- Weight loss due to inactivity and inability to manage blood sugar levels
- High cholesterol (hypercholesterolemia)
- Abnormal blood sugar (hyperglycemia)
- High blood pressure (hypertension)
- Sore and tender feet/legs (neurop

Q: How can I treat a mild fever at home?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Mild fever at home can be treated with appropriate measures, but it is essential to follow the below tips:

1. Drink plenty of fluids: Drinking plenty of fluids, water, and electrolytes, such as coconut water, lemon juice, or electrolyte drinks, can help lower the fever.
2. Stay cool: Staying cool in the summer can help lower the fever, as sweating reduces fever. Avoid extreme temperatures and wear loose and lightweight clothes.
3. Sleep: Getting more than 7-8 hours of sleep a night can help lower the fever.
4. Limit intake of caff

Q: What should I do if I have chest pain?
A: Chest pain is a symptom, and you need to consult a doctor regarding chest pain. If the pain is persistent, sudden, or severe, consult a doctor immediately. Chest pain may indicate a heart attack. Do not delay in seeking medical help. Chest pain in the left lower chest will be more than the right lower chest. If you have any chest pain, call 911 (emergency) or go to nearest emergency center immediately.
